# 01 — EEG Preprocessing

Loads Helsinki Neonatal EEG (3 EDF files: eeg10, eeg12, eeg14), applies
bandpass/notch filtering via MNE, extracts 30-second epochs with 50% overlap,
computes per-subject feature vectors (band power, BSR, IBI, SEF95), and saves
both tabular features and raw epoch arrays to `datasets/processed/eeg/`.

**run order: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9**

In [ ]:
# CELL 1: Clone / pull repo
import os
if not os.path.exists('/content/earlyMind'):
    !git clone https://github.com/Rickykapoor/earlyMind.git /content/earlyMind
%cd /content/earlyMind
!git pull origin main

In [ ]:
# CELL 2: Install dependencies
!pip install -q mne nibabel openneuro-py torch torchvision \
  plotly scikit-learn tqdm pyyaml scipy

In [ ]:
# CELL 3: Check GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
# EEG preprocessing is CPU-bound, GPU not required for this notebook

In [ ]:
# CELL 4: Data Loader (Supports DVC on Mounted Google Drive, Drag-and-Drop, or Auto Download)
import os, glob, shutil
os.makedirs('datasets/eeg/helsinki_neonatal', exist_ok=True)

# OPTION A: If your DVC cache is in your own Google Drive, point DVC to your mounted drive!
# => Update this to the exact folder path of your DVC storage inside your Google Drive
MY_DRIVE_DVC_PATH = '/content/drive/MyDrive/DVC/Earlyminds/earlyMind_DVC' # <-- Updated to your exact folder!

if os.path.exists(MY_DRIVE_DVC_PATH):
    print(f'Found local DVC remote at {MY_DRIVE_DVC_PATH}...')
    os.system(f'dvc remote add -d local_gdrive {MY_DRIVE_DVC_PATH} --force')
    os.system('dvc pull')
    print('DVC pull complete from mounted Google Drive!')

# OPTION B: Check if user dragged-and-dropped the raw files directly to /content/
if len(glob.glob('/content/*.edf')) > 0:
    print('Found raw files in /content/! Moving them to datasets/eeg/helsinki_neonatal...')
    os.system(f'mv /content/*.edf datasets/eeg/helsinki_neonatal/')
    os.system(f'mv /content/*.csv datasets/eeg/helsinki_neonatal/ 2>/dev/null')

# 2. Check if user dragged-and-dropped the .tar.gz file to /content/
elif os.path.exists('/content/eeg_raw.tar.gz'):
    print('Found eeg_raw.tar.gz in /content/! Extracting...')
    os.system(f'tar -xzf /content/eeg_raw.tar.gz -C datasets/eeg')

# 3. Fallback: Automatically download from GitHub Releases
else:
    print('Downloading dataset automatically from GitHub Releases...')
    os.system(f'wget -qO eeg_raw.tar.gz https://github.com/Rickykapoor/earlyMind/releases/download/v1.0.0-data/eeg_raw.tar.gz')
    os.system(f'tar -xzf eeg_raw.tar.gz -C datasets/eeg')

print('Datasets ready.')


In [ ]:
# CELL 5: Set up paths
import sys
sys.path.insert(0, '/content/earlyMind')

from src.config import cfg
from src.data.eeg_loader import process_eeg_dataset
import pandas as pd
import numpy as np

print('EEG raw dir: ', cfg.paths.eeg_raw)
print('EEG proc dir:', cfg.paths.eeg_processed)
cfg.paths.makedirs()

In [ ]:
# CELL 6: Inspect clinical CSV
# Actual columns: ID | EEG file | Gender | BW (g) | GA (weeks) | ... | Diagnosis | ...
clinical_path = cfg.paths.eeg_raw / 'clinical_information.csv'
ann_path = cfg.paths.eeg_raw / 'annotations_2017_A.csv'

df_clin = pd.read_csv(clinical_path)
print('Clinical CSV columns:', df_clin.columns.tolist())
print(f'Total subjects in CSV: {len(df_clin)}')
print('Subjects with EDF files available: 10, 12, 14')
display(df_clin[df_clin['ID'].isin([10, 12, 14])][['ID', 'EEG file', 'Diagnosis']])

# Annotations: WIDE matrix — columns = subject IDs, values = seizure flags per reviewer
df_ann = pd.read_csv(ann_path)
# Count subjects with any seizure (column sum > 0)
seizure_subjects = [c for c in df_ann.columns if pd.to_numeric(df_ann[c], errors='coerce').sum() > 0]
print(f'\nAnnotations: {df_ann.shape[0]} reviewers x {df_ann.shape[1]} subjects')
print(f'Subjects with annotations sum > 0: {len(seizure_subjects)} (incl. IDs: {sorted(seizure_subjects)[:10]}...)')
print(f'Of our 3 subjects — 10: {"10" in seizure_subjects}, 12: {"12" in seizure_subjects}, 14: {"14" in seizure_subjects}')

In [ ]:
# CELL 7: Run full EEG preprocessing
# NOTE: Each EDF file is ~40-58MB and takes ~5-15 min to process (Welch's method per epoch)
results = process_eeg_dataset(
    eeg_dir=cfg.paths.eeg_raw,
    output_dir=cfg.paths.eeg_processed,
)

print('\n=== Results ===')
for sid, info in results.items():
    print(f'  {sid}: epochs={info["epochs"].shape}, features={info["features"].shape}, '
          f'label={info["label"]}, DQ={info["dq"]:.1f}')

In [ ]:
# CELL 8: Quick sanity plot — feature distributions
import matplotlib.pyplot as plt

feat_names = ['Delta', 'Theta', 'Alpha', 'Beta', 'Total Power',
              'BSR', 'IBI Mean', 'IBI Std', 'SEF95', 'Amp Mean', 'Amp Std']

all_feats = np.stack([info['features'] for info in results.values()])
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
axes = axes.flatten()
for i, name in enumerate(feat_names):
    color = ['#ef4444' if info['label']==1 else '#22c55e' for info in results.values()]
    axes[i].bar(list(results.keys()), all_feats[:, i], color=color)
    axes[i].set_title(name, fontsize=9)
    axes[i].tick_params(axis='x', labelsize=7)
for ax in axes[len(feat_names):]:
    ax.set_visible(False)
plt.suptitle('EEG Features (red=ID risk, green=typical)', fontsize=12)
plt.tight_layout()
cfg.paths.reports.mkdir(parents=True, exist_ok=True)
plt.savefig(cfg.paths.reports / 'eeg_features.png', dpi=100)
plt.show()
print('labels.csv already saved by process_eeg_dataset.')
display(pd.read_csv(cfg.paths.eeg_processed / 'labels.csv'))

In [ ]:
# CELL 9: Commit and push
!git add checkpoints/ reports/ datasets/processed/
!git commit -m 'colab: 01_eeg_preprocess completed'
!git push origin main